# Comparativa de ASR usando METEOR

Este notebook calcula la métrica **METEOR** (Metric for Evaluation of Translation with Explicit ORdering) entre las transcripciones normalizadas (`text_normalized`) y las oraciones de referencia (ground truth).

METEOR considera coincidencias exactas, stem y sinónimos, y penaliza el orden. Se ejecuta **solo en entorno local** (sin Google Colab ni Drive).

In [10]:
# Instalar dependencias (solo uso local)
!pip install nltk pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 4.2 MB/s eta 0:00:00a 0:00:010m

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [11]:
import pandas as pd
import json
import os
import nltk
from nltk.translate.meteor_score import meteor_score

# Descargar recursos NLTK necesarios para METEOR (solo la primera vez)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print("METEOR: uso local.")

METEOR: uso local.


## 1. Cargar Datos

In [18]:
# Rutas locales (ejecución en máquina local)
dataset_path = 'normalized_dataset.csv'
ground_truth_path = 'ground_truth.json'

# Cargar el dataset normalizado
df = pd.read_csv(dataset_path)

# Cargar el ground truth
with open(ground_truth_path, 'r') as f:
    ground_truth = json.load(f)

# Crear un diccionario para mapear id -> texto de referencia
ref_dict = {item['id']: item['text'] for item in ground_truth}

# Verificar las primeras filas
df.head()

,person,audio,noise,snr,provider,text,status,transcription_time,text_normalized
0,p7,1,cafe,0dB,custom,Genera una cotización para el cliente con fáci...,success,1.65,genera 1 cotización para el cliente con fácil ...
1,p7,1,cafe,5dB,custom,Genera una cotización para el cliente con PUC ...,success,1.56,genera 1 cotización para el cliente con puc fa...
2,p7,1,cafe,10dB,custom,Genera una cotización para el cliente con Puff...,success,1.59,genera 1 cotización para el cliente con puffaz...
3,p7,1,clean,clean,custom,Genera una cotización para el cliente con FooF...,success,1.66,genera 1 cotización para el cliente con foofac...
4,p7,1,traffic,0dB,custom,"genera una cotización para el cliente fácil, c...",success,1.53,genera 1 cotización para el cliente fácil con ...


## 2. Preparar Candidatos y Referencias

In [19]:
# Asegurarse de que la columna 'audio' sea string para hacer el mapeo con el id del json
df['audio'] = df['audio'].astype(str)

# Obtener la lista de candidatos (transcripciones normalizadas)
# Rellenar NaNs con string vacío por si acaso
cands = df['text_normalized'].fillna('').tolist()

# Obtener la lista de referencias correspondientes usando la columna 'audio' (que es el ID)
refs = [ref_dict.get(audio_id, "") for audio_id in df['audio']]

# Verificar que tienen la misma longitud
assert len(cands) == len(refs), "Error: La longitud de candidatos y referencias no coincide."

print(f"Total de pares a evaluar: {len(cands)}")
print(f"Ejemplo candidato: {cands[0]}")
print(f"Ejemplo referencia: {refs[0]}")

Total de pares a evaluar: 6000
Ejemplo candidato: genera 1 cotización para el cliente con fácil con 5 monitores led y 3 soportes de pared
Ejemplo referencia: genera 1 cotización para el cliente compufacil con 5 monitores led y 3 soportes de pared


## 3. Calcular METEOR

Se calcula METEOR entre cada transcripción normalizada (hipótesis) y su oración de referencia. Ambas se tokenizan por palabras (split) y se usa la implementación de NLTK.

In [20]:
# Calcular METEOR para cada par (referencia, hipótesis)
# METEOR espera referencias como lista de listas de tokens, hipótesis como lista de tokens
def tokenize(s):
    return s.split() if s and str(s).strip() else ['']

scores = []
for ref, hyp in zip(refs, cands):
    ref_tok = tokenize(ref)
    hyp_tok = tokenize(hyp)
    if not ref_tok or ref_tok == [''] or not hyp_tok or hyp_tok == ['']:
        scores.append(0.0)
    else:
        try:
            score = meteor_score([ref_tok], hyp_tok)
            scores.append(score)
        except Exception:
            scores.append(0.0)

df['meteor_score'] = scores
print("Cálculo METEOR completado.")

Cálculo METEOR completado.


In [21]:
# Verificación rápida
print(f"Columnas con score: {[c for c in df.columns if 'meteor' in c.lower()]}")
print(f"Muestra de scores: {df['meteor_score'].head().tolist()}")

Columnas con score: ['meteor_score']
Muestra de scores: [0.9305728088336784, 0.9248285322359396, 0.9305728088336784, 0.9305728088336784, 0.6225]


## 4. Analizar Resultados

In [22]:
# Mostrar estadísticas descriptivas de los scores
print("Estadísticas de METEOR:")
print(df['meteor_score'].describe())

# Mostrar promedio agrupado por proveedor (provider)
if 'provider' in df.columns:
    print("\nPromedio de METEOR por proveedor:")
    print(df.groupby('provider')['meteor_score'].mean().sort_values(ascending=False))

# Mostrar promedio agrupado por nivel de ruido (noise)
if 'noise' in df.columns:
    print("\nPromedio de METEOR por tipo de ruido:")
    print(df.groupby('noise')['meteor_score'].mean().sort_values(ascending=False))

# Mostrar promedio agrupado por SNR
if 'snr' in df.columns:
    print("\nPromedio de METEOR por nivel de ruido (SNR):")
    print(df.groupby('snr')['meteor_score'].mean().sort_values(ascending=False))

Estadísticas de METEOR:
count    6000.000000
mean        0.942136
std         0.104579
min         0.000000
25%         0.922810
50%         0.999500
75%         0.999818
max         0.999878
Name: meteor_score, dtype: float64

Promedio de METEOR por proveedor:
provider
google    0.958295
custom    0.948286
azure     0.932086
amazon    0.929877
Name: meteor_score, dtype: float64

Promedio de METEOR por tipo de ruido:
noise
clean        0.976784
cafe         0.948543
traffic      0.944500
warehouse    0.921817
Name: meteor_score, dtype: float64

Promedio de METEOR por nivel de ruido (SNR):
snr
clean    0.976784
10dB     0.969849
5dB      0.956938
0dB      0.888073
Name: meteor_score, dtype: float64


In [23]:
# Guardar el dataframe con los scores (solo local)
output_filename = 'meteor_dataset.csv'
df.to_csv(output_filename, index=False)
print(f"Resultados guardados en {output_filename}")

Resultados guardados en meteor_dataset.csv


In [29]:
# Mostrar los 20 registros con menor METEOR
peores_20 = df.nsmallest(20, 'meteor_score')
display(peores_20[['audio', 'provider', 'snr', 'text_normalized', 'meteor_score']])

,audio,provider,snr,text_normalized,meteor_score
4467,12,google,0dB,3115370922078366001,0.000000
3714,12,google,0dB,0922078366001,0.042373
2247,15,custom,0dB,gracias por ver el video,0.052632
4797,15,azure,0dB,hundido durante no entiendo primero,0.052632
3637,4,google,0dB,quiero 1 persona para mi mejor,0.075758
5437,4,google,0dB,1 plataforma para mi motor,0.076336
3647,5,google,0dB,busca lo último en cultura y entretenimiento d...,0.106383
2447,5,custom,0dB,vista la última postura del presidente de la p...,0.107143
5434,4,google,0dB,mira 1 persona para mi laptop,0.113636
3657,6,google,0dB,pon la altura de 10 con 852015,0.126582


In [28]:
# Tabla: promedio de METEOR por proveedor y nivel de ruido
tabla_meteor = df.pivot_table(
    values='meteor_score',
    index='provider',
    columns='snr',
    aggfunc='mean'
).round(4)
display(tabla_meteor)

snr,0dB,10dB,5dB,clean
provider,,,,
amazon,0.8902,0.9509,0.9429,0.9468
azure,0.8717,0.9634,0.9460,0.9774
custom,0.8908,0.9771,0.9638,0.9880
google,0.8996,0.9880,0.9751,0.9949
